# `02-data-cleaning-and-exploration.ipynb`: Cleaning the scraped Arrests and Arrests & Incidents Logs and doing some preliminary data exploration

**Written by**: Diego Torrealba

**Project**: umpd-crime

**Date**: 2026-11-07

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from pathlib import Path

BASE_DIR = Path.cwd().parent
RAW_DATA_DIR = BASE_DIR / "data" / "raw" 
CURRENT_YEAR = time.localtime().tm_year
CURRENT_DAY = pd.to_datetime(f'{CURRENT_YEAR}-{time.localtime().tm_mon}-{time.localtime().tm_mday}')

# for time of day categorization
## 0 = 12 am; 23:59 - 11:59 pm
TOD_CATEGORIES = {
  'bins' : [0, 5, 11, 17, 23, 24],
  'labels' : ['night', 'morning', 'afternoon', 'evening', 'night']
}


# Semester start and end dates
SEMESTER_RANGES = {
  # 2017-2018
  'W_2018' : {'start_date': pd.to_datetime('2018-01-02'), 'end_date': pd.to_datetime('2018-01-22')},
  'SP_2018' : {'start_date': pd.to_datetime('2018-01-24'), 'end_date': pd.to_datetime('2018-05-18')},
  'SU_2018' : {'start_date': pd.to_datetime('2018-05-29'), 'end_date': pd.to_datetime('2018-08-17')},
  
  # 2018-2019
  'F_2018' : {'start_date': pd.to_datetime('2018-08-27'), 'end_date': pd.to_datetime('2018-12-18')},
  'W_2019' : {'start_date': pd.to_datetime('2019-01-02'), 'end_date': pd.to_datetime('2019-01-22')},
  'SP_2019' : {'start_date': pd.to_datetime('2019-01-28'), 'end_date': pd.to_datetime('2019-05-22')},
  'SU_2019' : {'start_date': pd.to_datetime('2019-05-28'), 'end_date': pd.to_datetime('2019-08-16')},
  
  # 2019-2020
  'F_2019' : {'start_date': pd.to_datetime('2019-08-26'), 'end_date': pd.to_datetime('2019-12-17')},
  'W_2020' : {'start_date': pd.to_datetime('2020-01-02'), 'end_date': pd.to_datetime('2020-01-22')},
  'SP_2020' : {'start_date': pd.to_datetime('2020-01-27'), 'end_date': pd.to_datetime('2020-05-20')},
  'SU_2020' : {'start_date': pd.to_datetime('2020-06-01'), 'end_date': pd.to_datetime('2020-08-21')},
  
  # 2020-2021
  'F_2020' : {'start_date': pd.to_datetime('2020-08-31'), 'end_date': pd.to_datetime('2020-12-22')},
  'W_2021' : {'start_date': pd.to_datetime('2021-01-04'), 'end_date': pd.to_datetime('2021-01-22')},
  'SP_2021' : {'start_date': pd.to_datetime('2021-01-25'), 'end_date': pd.to_datetime('2021-05-19')},
  'SU_2021' : {'start_date': pd.to_datetime('2021-06-01'), 'end_date': pd.to_datetime('2021-08-20')},
  
  # 2021-2022
  'F_2021' : {'start_date': pd.to_datetime('2021-08-30'), 'end_date': pd.to_datetime('2021-12-21')},
  'W_2022' : {'start_date': pd.to_datetime('2022-01-03'), 'end_date': pd.to_datetime('2022-01-21')},
  'SP_2022' : {'start_date': pd.to_datetime('2022-01-24'), 'end_date': pd.to_datetime('2022-05-18')},
  'SU_2022' : {'start_date': pd.to_datetime('2022-05-31'), 'end_date': pd.to_datetime('2022-08-19')},
  
  # 2022-2023
  'F_2022' : {'start_date': pd.to_datetime('2022-08-29'), 'end_date': pd.to_datetime('2022-12-20')},
  'W_2023' : {'start_date': pd.to_datetime('2023-01-03'), 'end_date': pd.to_datetime('2023-01-23')},
  'SP_2023' : {'start_date': pd.to_datetime('2023-01-25'), 'end_date': pd.to_datetime('2023-05-19')},
  'SU_2023' : {'start_date': pd.to_datetime('2023-05-30'), 'end_date': pd.to_datetime('2023-08-18')},
  
  # 2023-2024
  'F_2023' : {'start_date': pd.to_datetime('2023-08-28'), 'end_date': pd.to_datetime('2023-12-19')},
  'W_2024' : {'start_date': pd.to_datetime('2024-01-02'), 'end_date': pd.to_datetime('2024-01-22')},
  'SP_2024' : {'start_date': pd.to_datetime('2024-01-24'), 'end_date': pd.to_datetime('2024-05-17')},
  'SU_2024' : {'start_date': pd.to_datetime('2024-05-28'), 'end_date': pd.to_datetime('2024-08-16')},
  
  # 2024-2025
  'F_2024' : {'start_date': pd.to_datetime('2024-08-26'), 'end_date': pd.to_datetime('2024-12-17')},
  'W_2025' : {'start_date': pd.to_datetime('2025-01-02'), 'end_date': pd.to_datetime('2025-01-22')},
  'SP_2025' : {'start_date': pd.to_datetime('2025-01-27'), 'end_date': pd.to_datetime('2025-05-21')},
  'SU_2025' : {'start_date': pd.to_datetime('2025-06-02'), 'end_date': pd.to_datetime('2025-08-22')},
  
  # 2025-2026
  'F_2025' : {'start_date': pd.to_datetime('2025-09-02'), 'end_date': pd.to_datetime('2025-12-20')},
  'W_2026' : {'start_date': pd.to_datetime('2026-01-05'), 'end_date': pd.to_datetime('2026-01-23')},
  'SP_2026' : {'start_date': pd.to_datetime('2026-01-26'), 'end_date': pd.to_datetime('2026-05-18')},
  'SU_2026' : {'start_date': pd.to_datetime('2026-06-01'), 'end_date': pd.to_datetime('2026-08-21')},
  
  # 2026-now
  'F_2026': {'start_date': pd.to_datetime('2026-08-31'), 'end_date': CURRENT_DAY}
}

## SEMESTER_CONDITIONS
SEMESTER_LABELS = ['Winter 2018', 'Spring 2018', 'Summer 2018', 'Fall 2018', 'Winter 2019', 'Spring 2019', 'Summer 2019', 'Fall 2019', 'Winter 2020', 'Spring 2020', 'Summer 2020', 'Fall 2020', 'Winter 2021', 'Spring 2021', 'Summer 2021', 'Fall 2021', 'Winter 2022', 'Spring 2022', 'Summer 2022', 'Fall 2022', 'Winter 2023', 'Spring 2023', 'Summer 2023', 'Fall 2023', 'Winter 2024', 'Spring 2024', 'Summer 2024', 'Fall 2024', 'Winter 2025', 'Spring 2025', 'Summer 2025', 'Fall 2025', 'Winter 2026', 'Spring 2026', 'Summer 2026', 'Fall 2026']

def get_semester_conditions(series):
  """Returns list of Boolean conditions to set semester time frames, written by GPT-6.1 Sol on 2026-11-06 (stumped on dynamic condition creation)"""
  
  dates = pd.to_datetime(series).dt.normalize()
  
  return [
    dates.between(
      semester['start_date'],
      semester['end_date'],
      inclusive="both"
    )
    for semester in SEMESTER_RANGES.values()
  ]
  
def get_semester_season_conditions(series):
  
  series = pd.Series(series)
  
  return [series.str.startswith("Fall"),
          series.str.startswith("Winter"),
          series.str.startswith("Spring"),
          series.str.startswith("Summer")]

arrests_df = pd.read_csv(RAW_DATA_DIR / "arrest-ledger" / f"umpd_arrest_ledger_2018-{CURRENT_YEAR}.csv")
dci_df = pd.read_csv(RAW_DATA_DIR / 'daily-crime-incident-logs' / f'daily_crime_incident_log_2019-{CURRENT_YEAR}.csv')

In [2]:
CURRENT_DAY

Timestamp('2026-11-08 00:00:00')

## Arrests Log

In [3]:
arrests_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3728 entries, 0 to 3727
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   arrest_number     3728 non-null   int64  
 1   arrest_date       3728 non-null   str    
 2   umpd_case_number  3728 non-null   str    
 3   age               27 non-null     float64
 4   race              3650 non-null   str    
 5   sex               3651 non-null   str    
 6   arrested_charge   3728 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 204.0 KB


The columns that have values for every row are: `arrest_number`, `arrest_date`, `umpd_case_number` and `arrested_charge`. 

The columns that have missing values by order of how many they're missing: 

In [4]:
arrests_df.isna().sum()

arrest_number          0
arrest_date            0
umpd_case_number       0
age                 3701
race                  78
sex                   77
arrested_charge        0
dtype: int64

This checks out, as `age` is absent from most rows on the website. 

### Fixing datatypes

In [5]:
arrests_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3728 entries, 0 to 3727
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   arrest_number     3728 non-null   int64  
 1   arrest_date       3728 non-null   str    
 2   umpd_case_number  3728 non-null   str    
 3   age               27 non-null     float64
 4   race              3650 non-null   str    
 5   sex               3651 non-null   str    
 6   arrested_charge   3728 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 204.0 KB


- `arrest_number` should be a string, as its an identifier;
- `arrest_date` should be a datetime object

In [6]:
# Converting arrests_df columns to appropriate data types
arrests_df['arrest_number'] = arrests_df['arrest_number'].astype(str)
arrests_df['arrest_date'] = pd.to_datetime(arrests_df['arrest_date'])

### Creating supplementary columns from given columns

In [7]:
arrests_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3728 entries, 0 to 3727
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   arrest_number     3728 non-null   str           
 1   arrest_date       3728 non-null   datetime64[us]
 2   umpd_case_number  3728 non-null   str           
 3   age               27 non-null     float64       
 4   race              3650 non-null   str           
 5   sex               3651 non-null   str           
 6   arrested_charge   3728 non-null   str           
dtypes: datetime64[us](1), float64(1), str(5)
memory usage: 204.0 KB


In [8]:
arrests_df['time_of_day'] = pd.cut(
  arrests_df['arrest_date'].dt.hour,
  bins=TOD_CATEGORIES['bins'],
  labels=TOD_CATEGORIES['labels'],
  ordered=False,
  include_lowest=True
)
arrests_df[['arrest_date', 'time_of_day']].sample(10)

,arrest_date,time_of_day
547,2019-04-06 02:41:00,night
501,2019-03-12 10:00:00,morning
2879,2025-02-20 17:39:00,afternoon
1310,2020-10-08 22:35:00,evening
1173,2020-03-09 00:41:00,night
801,2019-08-06 00:00:00,night
3344,2026-02-22 01:10:00,night
138,2018-05-14 11:12:00,morning
2775,2024-12-05 01:31:00,night
2476,2024-03-22 13:13:00,afternoon


In [9]:
arrests_df['time_of_day'].value_counts(sort=True)

time_of_day
night        1961
evening       761
afternoon     734
morning       272
Name: count, dtype: int64

In [10]:
arrests_df[arrests_df['time_of_day'] == 'night']['arrest_date'].sample(10)

264    2018-09-21 04:02:00
165    2018-06-23 00:00:00
2142   2023-03-05 02:38:00
3429   2026-04-19 00:16:00
704    2019-06-14 01:57:00
348    2018-12-01 00:00:00
2689   2024-09-13 00:41:00
3175   2025-10-17 04:43:00
1998   2022-09-14 03:07:00
929    2019-10-05 01:15:00
Name: arrest_date, dtype: datetime64[us]

In [11]:
arrests_df_sem_conditions = get_semester_conditions(arrests_df['arrest_date'])
arrests_df['arrest_date_semester'] = np.select(arrests_df_sem_conditions, SEMESTER_LABELS, default = "Not during a semester")
arrests_df_sem_season_conditions = get_semester_season_conditions(arrests_df['arrest_date_semester'])
arrests_df['arrest_date_semester_season'] = np.select(
  arrests_df_sem_season_conditions,
  ['Fall', 'Winter', 'Spring', "Summer"],
  default = "Not during a semester"
)
arrests_df['arrest_date_semester_season'].value_counts()

arrest_date_semester_season
Spring                   1299
Fall                     1166
Summer                    755
Not during a semester     354
Winter                    154
Name: count, dtype: int64

### Splitting charges

In [12]:
arrests_df['arrested_charges'] = arrests_df['arrested_charge'].str.split(';')
arrests_df['arrested_charges_num'] = arrests_df['arrested_charges'].apply(lambda x: len(x))
arrests_df = arrests_df[['arrest_number', 'umpd_case_number', 'arrest_date', 'arrest_date_semester', 'arrest_date_semester_season', 'age', 'race', 'sex', 'arrested_charge', 'arrested_charges_num', 'arrested_charges']]
arrests_df_exploded = arrests_df.explode('arrested_charges')
arrests_df_exploded['arrested_charges'] = arrests_df_exploded['arrested_charges'].str.strip()

In [13]:
arrests_df_exploded['arrested_charges_num'].describe()

count    8715.000000
mean        4.089845
std         2.813307
min         1.000000
25%         2.000000
50%         3.000000
75%         6.000000
max        15.000000
Name: arrested_charges_num, dtype: float64

In [14]:
arrests_df_exploded[arrests_df_exploded['arrested_charges_num'] == 15]

,arrest_number,umpd_case_number,arrest_date,arrest_date_semester,arrest_date_semester_season,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,"(Driving, Attempting to drive) veh. while impa..."
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,"(Driving, Attempting to drive) veh. while unde..."
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,"(Driving, Attempting to drive) veh. while unde..."
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driver changing lanes when unsafe
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driver failure to obey properly placed traffic...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driver failure to sound horn warning to pedest...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driver failure to stop for pedestrian in cross...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driving veh. in excess of reasonable and prude...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Driving veh. on hwy. at speed exceeding speed ...
1725,23304,2021-00093620,2021-12-12 01:24:00,Fall 2021,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",15,Exceeding highway work zone speed limit


In [15]:
arrests_df_exploded['arrested_charges'].value_counts(sort=True).head(15)

arrested_charges
(Driving, Attempting to drive) veh. while under the influence of alcohol                                             1281
(Driving, Attempting to drive) veh. while impaired by alcohol                                                        1172
(Driving, Attempting to drive) veh. while under the influence of alcohol per se                                       730
Reckless driving veh. in wanton and willfull disregard for safety of persons and property                             488
CDS:Possession Marijuana L/T 10 grams                                                                                 457
Negligent driving veh. in careless and imprudent manner endangering property, life and person                         457
School Trespassing On Ground                                                                                          165
THEFT: $100 TO UNDER $1,500-All Other                                                                                 153
Driver 

In [16]:
len(arrests_df_exploded['arrested_charges'].unique().tolist())

442

I gave GPT-6 Astra Light the list of 442 unique charges and asked it to categorize them.

In [17]:
arrests_df_gpt_categories = pd.read_csv(RAW_DATA_DIR.parent / 'processed' / 'gpt_charge_categories.csv')
arrests_df_exploded = pd.merge(arrests_df_exploded, arrests_df_gpt_categories, 'left', left_on='arrested_charges', right_on="charge_label")
arrests_df_exploded = arrests_df_exploded.rename({'category': 'gpt_category'}, axis=1)

I also gave Claude Sonnet 5.5 High the list of charges to see how the categories between the two models differed.

In [18]:
arrests_df_exploded

,arrest_number,umpd_case_number,arrest_date,arrest_date_semester,arrest_date_semester_season,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges,source_position,charge_label,gpt_category
0,21483,2018-00000017,2018-01-01 02:15:00,Not during a semester,Not during a semester,NaN,NaN,NaN,"(Driving, Attempting to drive) veh. while impa...",2,"(Driving, Attempting to drive) veh. while impa...",1,"(Driving, Attempting to drive) veh. while impa...",Impaired driving and related driving restrictions
1,21483,2018-00000017,2018-01-01 02:15:00,Not during a semester,Not during a semester,NaN,NaN,NaN,"(Driving, Attempting to drive) veh. while impa...",2,"(Driving, Attempting to drive) veh. while unde...",2,"(Driving, Attempting to drive) veh. while unde...",Impaired driving and related driving restrictions
2,21484,2017-00078928,2018-01-03 14:35:00,Winter 2018,Winter,20.0,Black,Male,"THEFT: $100 TO UNDER $1,500",1,"THEFT: $100 TO UNDER $1,500",3,"THEFT: $100 TO UNDER $1,500","Theft, theft schemes and unauthorized use or r..."
3,21485,2018-00001737,2018-01-09 12:02:00,Winter 2018,Winter,NaN,White,Male,Assault-1st Degree; Other Dangerous Weapon; Da...,3,Assault-1st Degree,4,Assault-1st Degree,"Assault, endangerment and unlawful restraint"
4,21485,2018-00001737,2018-01-09 12:02:00,Winter 2018,Winter,NaN,White,Male,Assault-1st Degree; Other Dangerous Weapon; Da...,3,Other Dangerous Weapon,5,Other Dangerous Weapon,Source descriptors and incomplete labels
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8710,25182,2026-00289290,2026-11-01 03:17:00,Fall 2026,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",3,"(Driving, Attempting to drive) veh. while unde...",2,"(Driving, Attempting to drive) veh. while unde...",Impaired driving and related driving restrictions
8711,25182,2026-00289290,2026-11-01 03:17:00,Fall 2026,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",3,"(Driving, Attempting to drive) veh. while unde...",16,"(Driving, Attempting to drive) veh. while unde...",Impaired driving and related driving restrictions
8712,25188,2026-00289287,2026-11-01 03:33:00,Fall 2026,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",3,"(Driving, Attempting to drive) veh. while impa...",1,"(Driving, Attempting to drive) veh. while impa...",Impaired driving and related driving restrictions
8713,25188,2026-00289287,2026-11-01 03:33:00,Fall 2026,Fall,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",3,"(Driving, Attempting to drive) veh. while unde...",2,"(Driving, Attempting to drive) veh. while unde...",Impaired driving and related driving restrictions


In [19]:
arrests_df_exploded = arrests_df_exploded[['arrest_number', 'umpd_case_number', 'arrest_date', 'arrest_date_semester', 'arrest_date_semester_season', 'age', 'race', 'sex', 'arrested_charge', 'arrested_charges_num', 'arrested_charges', 'gpt_category']]
arrests_df_exploded['gpt_category'].value_counts()

gpt_category
Impaired driving and related driving restrictions                                   3389
Moving violations and road-user safety                                              1482
Theft, theft schemes and unauthorized use or removal of property                     780
Drug possession, use and paraphernalia (including cannabis)                          684
Licensing, registration and insurance                                                275
Trespassing and unlawful entry                                                       260
Property damage, burning and tampering                                               211
Burglary, vehicle break-ins and burglary tools                                       197
Alcohol possession, consumption, furnishing and public intoxication                  179
Assault, endangerment and unlawful restraint                                         170
Fraud, forgery, false identification and financial offenses                          165
Source d

## Daily Crime Incidents and Arrests

In [20]:
dci_df.head()

,umpd_case_number,date_occurred,report_date,crime_type,disposition,location
0,2019-00000001,01/01/2019 00:01:00,01/01/2019 - 00:01,Fireworks Complaint,Arrest,4300 block of Knox Rd
1,2019-00000009,01/01/2019 01:20:00,01/01/2019 - 01:20,DWI/DUI,Arrest,8300 block of Baltimore Ave
2,2019-00000011,01/01/2019 01:28:00,01/01/2019 - 01:28,DWI/DUI,Arrest,7400 block of Baltimore Ave
3,2019-00000203,12/28/2018 17:00:00,01/02/2019 - 11:34,Vandalism,CBE,7500 block of Calvert Service Ln
4,2019-00000312,01/02/2019 23:04:00,01/02/2019 - 23:04,CDS Violation,Arrest,Metzerott Rd


In [21]:
dci_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12912 entries, 0 to 12911
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   umpd_case_number  12912 non-null  str  
 1   date_occurred     12912 non-null  str  
 2   report_date       12912 non-null  str  
 3   crime_type        12912 non-null  str  
 4   disposition       12912 non-null  str  
 5   location          12866 non-null  str  
dtypes: str(6)
memory usage: 605.4 KB


- `date_occurred` and `report_date` should be datetime objects.

In [22]:
# Around 51 out of over 12000 have no time stamps. Setting them to midnight
dci_df.loc[dci_df['date_occurred'].str.contains(":") != True, 'date_occurred'] = dci_df.loc[dci_df['date_occurred'].str.contains(":") != True, 'date_occurred'] + " 00:00:00"

dci_df['date_occurred'] = pd.to_datetime(dci_df['date_occurred'])
dci_df['report_date'] = pd.to_datetime(dci_df['report_date'])

In [23]:
dci_df.head()

,umpd_case_number,date_occurred,report_date,crime_type,disposition,location
0,2019-00000001,2019-01-01 00:01:00,2019-01-01 00:01:00,Fireworks Complaint,Arrest,4300 block of Knox Rd
1,2019-00000009,2019-01-01 01:20:00,2019-01-01 01:20:00,DWI/DUI,Arrest,8300 block of Baltimore Ave
2,2019-00000011,2019-01-01 01:28:00,2019-01-01 01:28:00,DWI/DUI,Arrest,7400 block of Baltimore Ave
3,2019-00000203,2018-12-28 17:00:00,2019-01-02 11:34:00,Vandalism,CBE,7500 block of Calvert Service Ln
4,2019-00000312,2019-01-02 23:04:00,2019-01-02 23:04:00,CDS Violation,Arrest,Metzerott Rd


In [24]:
dci_df['date_occurred_tod'] = pd.cut(
  dci_df['date_occurred'].dt.hour,
  bins = TOD_CATEGORIES['bins'],
  labels = TOD_CATEGORIES['labels'],
  ordered = False,
  include_lowest = True
)

dci_df_date_occurred_sem_conditions = get_semester_conditions(dci_df['date_occurred'])
dci_df['date_occurred_sem'] = np.select(dci_df_date_occurred_sem_conditions, SEMESTER_LABELS, default = "Not during a semester")

dci_df_date_occurred_sem_season_conditions = get_semester_season_conditions(dci_df['date_occurred_sem'])

dci_df['date_occurred_sem_season'] = np.select(
  dci_df_date_occurred_sem_season_conditions,
  ['Fall', "Winter", "Spring", "Summer"],
  default = "Not during a semester"
)

In [25]:
dci_df['report_date_tod'] = pd.cut(
  dci_df['report_date'].dt.hour,
  bins = TOD_CATEGORIES['bins'],
  labels = TOD_CATEGORIES['labels'],
  ordered = False,
  include_lowest = True
)
dci_df_report_date_sem_conditions = get_semester_conditions(dci_df['report_date'])
dci_df['report_date_sem'] = np.select(dci_df_report_date_sem_conditions, SEMESTER_LABELS, default = "Not during a semester")

dci_df_report_date_sem_season_conditions = get_semester_season_conditions(dci_df['report_date_sem'])

dci_df['report_date_sem_season'] = np.select(
  dci_df_date_occurred_sem_season_conditions,
  ['Fall', 'Winter', 'Spring', 'Summer'],
  default= 'Not during a semester'
)

In [26]:
dci_df = dci_df[['umpd_case_number', 'date_occurred', 'date_occurred_tod', 'date_occurred_sem', 'date_occurred_sem_season', 'report_date', 'report_date_tod', 'report_date_sem', 'report_date_sem_season', 'crime_type', 'disposition', 'location']]

In [27]:
dci_df

,umpd_case_number,date_occurred,date_occurred_tod,date_occurred_sem,date_occurred_sem_season,report_date,report_date_tod,report_date_sem,report_date_sem_season,crime_type,disposition,location
0,2019-00000001,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,Fireworks Complaint,Arrest,4300 block of Knox Rd
1,2019-00000009,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,DWI/DUI,Arrest,8300 block of Baltimore Ave
2,2019-00000011,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,DWI/DUI,Arrest,7400 block of Baltimore Ave
3,2019-00000203,2018-12-28 17:00:00,afternoon,Not during a semester,Not during a semester,2019-01-02 11:34:00,morning,Winter 2019,Not during a semester,Vandalism,CBE,7500 block of Calvert Service Ln
4,2019-00000312,2019-01-02 23:04:00,evening,Winter 2019,Winter,2019-01-02 23:04:00,evening,Winter 2019,Winter,CDS Violation,Arrest,Metzerott Rd
...,...,...,...,...,...,...,...,...,...,...,...,...
12907,2026-00289088,2026-10-31 20:28:00,evening,Fall 2026,Fall,2026-10-31 20:28:00,evening,Fall 2026,Fall,Injured/Sick Person,CBE,"Frat 2, Sigma Alpha Mu, Sigma Alpha Mu Fratern..."
12908,2026-00289157,2026-10-31 23:56:00,evening,Fall 2026,Fall,2026-10-31 23:56:00,evening,Fall 2026,Fall,DWI/DUI,Arrest,Baltimore Ave
12909,2026-00289166,2026-11-01 00:08:00,night,Fall 2026,Fall,2026-11-01 00:08:00,night,Fall 2026,Fall,Injured/Sick Person,CBE,8600 block of Boteler Ln
12910,2026-00289167,2026-11-01 00:25:00,night,Fall 2026,Fall,2026-11-01 00:25:00,night,Fall 2026,Fall,Injured/Sick Person,CBE,3900 block of Campus Dr


## Merging detailed datasets

In [28]:
merged_dfs = pd.merge(dci_df, arrests_df, "left", on="umpd_case_number")

In [30]:
merged_dfs.to_csv(Path(RAW_DATA_DIR.parent / 'processed' / f"merged_arrest_ci_data_2018-{CURRENT_YEAR}.csv"))